# Eclipse Phase Classifier: Training and Evaluation

**Requires:** `eclipse_measurements.xlsx` (produced by `Eclipse_pipeline.ipynb`) in the same folder.

```
pip install pandas scikit-learn openpyxl
```

Trains three classifiers (k-nearest-neighbors, logistic regression, decision tree) to predict eclipse phase
from the measured image features, evaluates each with a held-out test set and 5-fold cross-validation,
and runs an ablation test with the `Coverage_pct` feature removed to check whether the other measured
features (radius, fit error, sunspot count, luminosity, crescent geometry) carry independent signal.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DATA_PATH = "eclipse_measurements.xlsx"
df = pd.read_excel(DATA_PATH)
print(f"Loaded {len(df)} rows")
print(df["Phase_label"].value_counts())

## Feature sets

`FEATURES_FULL` includes coverage. `FEATURES_ABLATION` removes it, to test whether the other
measured features can predict phase on their own.

In [ ]:
FEATURES_FULL = ["Coverage_pct", "Fit_radius_px", "Radius_used_px", "Fit_RMS_px",
                 "Sunspots", "Crescent_angle_deg", "Centroid_offset", "Luminosity_norm"]
FEATURES_ABLATION = [c for c in FEATURES_FULL if c != "Coverage_pct"]

y = df["Phase_label"]

MODELS = {
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

def evaluate(features, label, random_state=42):
    X = df[features].fillna(0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    X_s = scaler.transform(X)
    print(f"\n=== {label} (n_features={len(features)}) ===")
    results = {}
    for name, clf in MODELS.items():
        clf.fit(Xtr_s, ytr)
        pred = clf.predict(Xte_s)
        acc = accuracy_score(yte, pred)
        cv = cross_val_score(clf, X_s, y, cv=StratifiedKFold(5, shuffle=True, random_state=random_state))
        results[name] = {"held_out_acc": acc, "cv_mean": cv.mean(), "model": clf, "pred": pred, "ytest": yte}
        print(f"{name:22s} held-out acc={acc:.3f}   5-fold CV mean={cv.mean():.3f}")
    return results

results_full = evaluate(FEATURES_FULL, "Full feature set (with coverage)")
results_ablation = evaluate(FEATURES_ABLATION, "Ablation (coverage removed)")

## Confusion matrix (KNN)

Using KNN here rather than whichever model scored highest: the Decision Tree hits 100% on the full
feature set, which (see the notes below) is an overfitting artifact of `Phase_label` being a fixed
threshold rule applied to `Coverage_pct`, not a meaningful classification result. KNN's accuracy is
nearly as high and isn't inflated the same way.

In [ ]:
best_name = "KNN (k=5)"
best = results_full[best_name]
print(f"Model: {best_name}\n")
labels = sorted(y.unique())
cm = confusion_matrix(best["ytest"], best["pred"], labels=labels)
print(pd.DataFrame(cm, index=labels, columns=labels))
print()
print(classification_report(best["ytest"], best["pred"]))

## Notes on interpreting these results

- `Phase_label` is generated from `Coverage_pct` by a fixed threshold rule in the measurement pipeline,
  so a model given `Coverage_pct` directly is partly re-deriving a known rule rather than performing
  independent inference. The Decision Tree reaching 100% held-out accuracy in the full feature set is
  expected for this reason, it can effectively learn the threshold boundaries, and should not be read
  as a meaningful result on its own.
- The ablation run (coverage removed) is the more meaningful test of whether the pipeline's other
  measured features (radius, fit error, sunspot count, luminosity, crescent geometry) carry real signal
  about eclipse phase on their own, independent of the coverage number itself.
- Results depend on the random train/test split and the exact image set used. Re-running with a
  different `random_state` or a different subset of frames will shift these numbers somewhat.